# ML-06 — Signal Audit: Do the Flags Hold?

This notebook tests descriptive signals on the active content corpus. All displayed values and verdicts are calculated by the code below; the results are observational and do not establish causation.

## 1. Load the active corpus

The contract keeps pages with at least 10 impressions in the rolling 90-day window and at least 90 days of content age. An average position of zero means no ranking data, so it is excluded from the position correlation only.

In [ ]:
import os
import numpy as np
import pandas as pd

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
df_clean = df_raw.loc[
    (df_raw['impressions_90d'] >= 10) & (df_raw['content_age_days'] >= 90)
] .copy()

print(f'Active corpus: {len(df_clean):,} pages across {df_clean["client_id"].nunique()} pseudonymized clients')


## 2. Distributions

The summary and histogram show whether a transformation is appropriate before a distance-based model is fitted.

In [ ]:
import matplotlib.pyplot as plt

distribution_columns = [
    'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'engagement_rate',
]
display(df_clean[distribution_columns].describe().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_clean['impressions_90d'], bins=50)
axes[0].set_title('Raw impressions')
axes[0].set_xlabel('Impressions (90d)')
axes[1].hist(np.log1p(df_clean['impressions_90d']), bins=50)
axes[1].set_title('log1p impressions')
axes[1].set_xlabel('log1p(impressions)')
plt.tight_layout()
plt.show()


## 3. Signal tests

Each verdict is calculated from a stated rule. Pearson correlation describes a linear association in this dataset; it is not a predictive model or causal estimate.

In [ ]:
corr_pos_ctr = df_clean['avg_position'].replace(0, np.nan).corr(df_clean['ctr'])
corr_fresh_eng = df_clean['days_since_last_update'].corr(df_clean['engagement_rate'])
imp_skew = df_clean['impressions_90d'].skew()

audit_results = pd.DataFrame({
    'Signal': [
        'Impression skew',
        'Position–CTR correlation',
        'Freshness–engagement correlation',
    ],
    'Value': [round(imp_skew, 3), round(corr_pos_ctr, 3), round(corr_fresh_eng, 3)],
    'Verdict': [
        'CONFIRMED' if imp_skew > 2 else 'MIXED',
        'CONFIRMED' if corr_pos_ctr < -0.3 else 'MIXED',
        'CONFIRMED' if corr_fresh_eng < -0.1 else 'MIXED',
    ],
})
display(audit_results)


## 4. Flag-linked test: high-impression and stale pages

The same condition is reported against two explicit denominators: the active corpus and high-impression pages. This prevents the two shares from being conflated.

In [ ]:
high_impression = df_clean[df_clean['impressions_90d'] >= 500]
stale_high_impression = high_impression[
    high_impression['days_since_last_update'] >= 180
]

stale_count = len(stale_high_impression)
pct_of_corpus = stale_count / len(df_clean) * 100
pct_of_high_impression = stale_count / len(high_impression) * 100

flag_summary = pd.DataFrame({
    'Measure': ['Stale high-impression pages', 'Share of active corpus', 'Share of high-impression pages'],
    'Value': [stale_count, round(pct_of_corpus, 3), round(pct_of_high_impression, 3)],
})
display(flag_summary)


## 5. Interpretation

The code above is the source of truth for the findings. If the position–CTR association is weak under the stated rule, position should be treated as one signal among several rather than as a standalone predictor. Likewise, the stale-page condition may be operationally useful while still representing a small share of either denominator.

## Self-check

- [x] Every verdict is generated from the current code.
- [x] Denominators are named explicitly for the stale-page condition.
- [x] Claims are framed as observed associations and decision-support.
- [x] No client names, URLs, or private queries are displayed.